# ⚡ Train Photorealistic Custom Hero Face Swap Models on Google Colab (Free T4 GPU)

This notebook trains a **DeepFaceLab-grade Shared Encoder + Hero Decoder** model on your chosen celebrity or hero's face crops with:
- **VGG19 Deep Perceptual Feature Loss**
- **Structural Similarity Index (SSIM) Loss**
- **Sobel High-Frequency Edge Gradient Loss**
- **Live Visual Preview Reconstruction Grids**
- **1-Click ONNX Export** directly usable in the Real-Time Face Swap Camera app (`python scripts/run_app.py`)

---
### 🚀 Step 0: Ensure GPU Acceleration is Enabled
In the menu bar above, click **Runtime** -> **Change runtime type** -> Select **T4 GPU** -> Click **Save**.

In [ ]:
# Check active GPU
!nvidia-smi

# Install dependencies
!pip install torch torchvision onnx opencv-python-headless matplotlib

### 📁 Step 1: Upload Your Hero Dataset
Upload the `hero_dataset.zip` generated on your PC using:
```bash
python scripts/package_training_dataset.py --input faces/actors/prabhas --output outputs/hero_dataset.zip
```
Run the cell below to upload:

In [ ]:
from google.colab import files
import zipfile, os

os.makedirs("dataset_crops", exist_ok=True)

print("Please select your hero_dataset.zip file:")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn, 'r') as zip_ref:
            zip_ref.extractall("dataset_crops")
        print(f"Extracted {fn} to dataset_crops/")

import glob
img_count = len(glob.glob("dataset_crops/**/*.*", recursive=True))
print(f"Successfully loaded {img_count} face crops for training!")

### 🧠 Step 2: Define Model Architecture & Perceptual Losses

In [ ]:
import os, glob, math, time, cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19, VGG19_Weights
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm2d(channels, affine=True)
        self.leaky = nn.LeakyReLU(0.2, inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm2d(channels, affine=True)

    def forward(self, x):
        return self.leaky(self.norm2(self.conv2(self.leaky(self.norm1(self.conv1(x)))))) + x

class ConvDownBlock(nn.Module):
    def __init__(self, in_c: int, out_c: int):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False)
        self.norm = nn.InstanceNorm2d(out_c, affine=True)
        self.leaky = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.leaky(self.norm(self.conv(x)))

class PixelShuffleUpscale(nn.Module):
    def __init__(self, in_c: int, out_c: int):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c * 4, kernel_size=3, padding=1, bias=False)
        self.pixel_shuffle = nn.PixelShuffle(2)
        self.norm = nn.InstanceNorm2d(out_c, affine=True)
        self.leaky = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.leaky(self.norm(self.pixel_shuffle(self.conv(x))))

class SharedEncoder(nn.Module):
    def __init__(self, in_channels: int = 3, latent_dim: int = 512):
        super().__init__()
        self.down1 = ConvDownBlock(in_channels, 64)    # 128 -> 64
        self.down2 = ConvDownBlock(64, 128)            # 64 -> 32
        self.down3 = ConvDownBlock(128, 256)           # 32 -> 16
        self.down4 = ConvDownBlock(256, 512)           # 16 -> 8
        self.res1 = ResidualBlock(512)
        self.res2 = ResidualBlock(512)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512 * 8 * 8, latent_dim)

    def forward(self, x):
        x = self.res2(self.res1(self.down4(self.down3(self.down2(self.down1(x))))))
        return self.fc(self.flatten(x))

class HeroDecoder(nn.Module):
    def __init__(self, latent_dim: int = 512, out_channels: int = 3):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 8 * 8)
        self.res1 = ResidualBlock(512)
        self.res2 = ResidualBlock(512)
        self.up1 = PixelShuffleUpscale(512, 256)        # 8 -> 16
        self.up2 = PixelShuffleUpscale(256, 128)        # 16 -> 32
        self.up3 = PixelShuffleUpscale(128, 64)         # 32 -> 64
        self.up4 = PixelShuffleUpscale(64, 32)          # 64 -> 128
        self.out_conv = nn.Conv2d(32, out_channels, kernel_size=3, padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, latent):
        x = F.leaky_relu(self.fc(latent), 0.2).view(-1, 512, 8, 8)
        x = self.up4(self.up3(self.up2(self.up1(self.res2(self.res1(x))))))
        return self.sigmoid(self.out_conv(x))

class FaceSwapModel(nn.Module):
    def __init__(self, latent_dim: int = 512):
        super().__init__()
        self.encoder = SharedEncoder(3, latent_dim)
        self.decoder = HeroDecoder(latent_dim, 3)

    def forward(self, source_face):
        return self.decoder(self.encoder(source_face))

# Losses: SSIM + Sobel + VGG19
class SSIMLoss(nn.Module):
    def __init__(self, window_size: int = 11, channels: int = 3):
        super().__init__()
        self.window_size = window_size
        self.channels = channels
        sigma = 1.5
        gauss = torch.Tensor([math.exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) for x in range(window_size)])
        gauss = gauss / gauss.sum()
        _1D = gauss.unsqueeze(1)
        _2D = _1D.mm(_1D.t()).float().unsqueeze(0).unsqueeze(0)
        self.register_buffer("window", _2D.expand(channels, 1, window_size, window_size).contiguous())

    def forward(self, img1, img2):
        window = self.window.to(img1.device)
        mu1 = F.conv2d(img1, window, padding=self.window_size // 2, groups=self.channels)
        mu2 = F.conv2d(img2, window, padding=self.window_size // 2, groups=self.channels)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        sig1_sq = F.conv2d(img1 * img1, window, padding=self.window_size // 2, groups=self.channels) - mu1_sq
        sig2_sq = F.conv2d(img2 * img2, window, padding=self.window_size // 2, groups=self.channels) - mu2_sq
        sig12 = F.conv2d(img1 * img2, window, padding=self.window_size // 2, groups=self.channels) - mu1_mu2
        ssim = ((2 * mu1_mu2 + 0.0001) * (2 * sig12 + 0.0009)) / ((mu1_sq + mu2_sq + 0.0001) * (sig1_sq + sig2_sq + 0.0009))
        return 1.0 - ssim.mean()

class SobelGradientLoss(nn.Module):
    def __init__(self, channels: int = 3):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).unsqueeze(0).unsqueeze(0).repeat(channels, 1, 1, 1)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).unsqueeze(0).unsqueeze(0).repeat(channels, 1, 1, 1)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)
        self.channels = channels

    def forward(self, p, t):
        return F.l1_loss(F.conv2d(p, self.kx.to(p.device), padding=1, groups=self.channels), F.conv2d(t, self.kx.to(t.device), padding=1, groups=self.channels)) + \
               F.l1_loss(F.conv2d(p, self.ky.to(p.device), padding=1, groups=self.channels), F.conv2d(t, self.ky.to(t.device), padding=1, groups=self.channels))

class VGGPerceptualLoss(nn.Module):
    def __init__(self, dev):
        super().__init__()
        vgg = vgg19(weights=VGG19_Weights.DEFAULT).features.to(dev).eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.s1 = nn.Sequential(*[vgg[x] for x in range(4)]).to(dev)
        self.s2 = nn.Sequential(*[vgg[x] for x in range(4, 9)]).to(dev)
        self.s3 = nn.Sequential(*[vgg[x] for x in range(9, 14)]).to(dev)
        self.s4 = nn.Sequential(*[vgg[x] for x in range(14, 23)]).to(dev)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(dev))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(dev))

    def forward(self, p, t):
        p_n = (p - self.mean) / self.std
        t_n = (t - self.mean) / self.std
        return 0.1 * F.l1_loss(self.s1(p_n), self.s1(t_n)) + \
               0.2 * F.l1_loss(self.s2(self.s1(p_n)), self.s2(self.s1(t_n))) + \
               0.5 * F.l1_loss(self.s3(self.s2(self.s1(p_n))), self.s3(self.s2(self.s1(t_n)))) + \
               1.0 * F.l1_loss(self.s4(self.s3(self.s2(self.s1(p_n)))), self.s4(self.s3(self.s2(self.s1(t_n)))))

print("Model architecture and loss functions initialized successfully!")

### ⚡ Step 3: Run High-Speed GPU Training
Recommended epochs: 200–300 epochs. On a free Google Colab T4 GPU, 200 epochs finishes in only **~5 to 8 minutes**!

In [ ]:
# Dataset Loader with live augmentations
class FaceDataset(Dataset):
    def __init__(self, root_dir, augment=True):
        self.paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
            self.paths.extend(glob.glob(os.path.join(root_dir, "**", ext), recursive=True))
        self.augment = augment

    def __len__(self):
        return max(1, len(self.paths))

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx % len(self.paths)])
        img = cv2.resize(img, (128, 128), interpolation=cv2.INTER_LANCZOS4)
        if self.augment:
            if np.random.rand() > 0.5:
                img = cv2.flip(img, 1)
            if np.random.rand() > 0.4:
                M = cv2.getRotationMatrix2D((64, 64), np.random.uniform(-5, 5), 1.0)
                img = cv2.warpAffine(img, M, (128, 128), borderMode=cv2.BORDER_REFLECT_101)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return torch.from_numpy(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1)

EPOCHS = 200
BATCH_SIZE = 16
LR = 2e-4

dataset = FaceDataset("dataset_crops", augment=True)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True if len(dataset.paths) > BATCH_SIZE else False)

sample_dataset = FaceDataset("dataset_crops", augment=False)
sample_loader = DataLoader(sample_dataset, batch_size=min(4, len(dataset.paths)), shuffle=False)
sample_batch = next(iter(sample_loader))

model = FaceSwapModel(latent_dim=512).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.5, 0.999), weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

criterion_l1 = nn.L1Loss()
criterion_ssim = SSIMLoss().to(device)
criterion_sobel = SobelGradientLoss().to(device)
criterion_vgg = VGGPerceptualLoss(device)

os.makedirs("checkpoints", exist_ok=True)
best_loss = float("inf")
t0 = time.time()

print(f"Starting GPU training for {EPOCHS} epochs on {len(dataset.paths)} face crops...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        recon = model(batch)
        loss = criterion_l1(recon, batch) + 0.5 * criterion_ssim(recon, batch) + 0.3 * criterion_sobel(recon, batch) + 0.2 * criterion_vgg(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / max(1, len(loader))

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "checkpoints/hero_model_best.pt")

    if epoch % 20 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] - Loss: {avg_loss:.4f} | Elapsed: {time.time()-t0:.1f}s")

print("\n[SUCCESS] Training Complete! Best model saved to checkpoints/hero_model_best.pt")

### 👁️ Step 4: Visualize Final Face Reconstruction Quality

In [ ]:
# Load best checkpoint and plot side-by-side
model.load_state_dict(torch.load("checkpoints/hero_model_best.pt", map_location=device))
model.eval()

with torch.no_grad():
    test_recon = model(sample_batch.to(device)).cpu()

fig, axes = plt.subplots(2, sample_batch.size(0), figsize=(12, 6))
for i in range(sample_batch.size(0)):
    orig = sample_batch[i].permute(1, 2, 0).numpy()
    rec = test_recon[i].permute(1, 2, 0).numpy()
    axes[0, i].imshow(orig)
    axes[0, i].set_title("Original Target")
    axes[0, i].axis("off")
    axes[1, i].imshow(rec)
    axes[1, i].set_title("Model Output")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

### 📦 Step 5: Export to ONNX & Download Model
Exports your trained hero model directly to `model.onnx`.

In [ ]:
# Export to ONNX
model.eval().to("cpu")
dummy_input = torch.randn(1, 3, 128, 128, dtype=torch.float32)
onnx_output_path = "model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_output_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["target"],
    output_names=["output"],
    dynamic_axes={"target": {0: "batch_size"}, "output": {0: "batch_size"}}
)

print(f"Successfully exported ONNX model ({round(os.path.getsize(onnx_output_path)/(1024*1024), 2)} MB)")

# Download the model to your PC
files.download(onnx_output_path)
print("Downloading model.onnx to your browser downloads folder...")
print("Move this file to: models/face_swap/model.onnx in your project!")